# Implementing Word2vec in PyTorch from the Ground Up

based on https://towardsdatascience.com/implementing-word2vec-in-pytorch-from-the-ground-up-c7fe5bf99889/


## Intro & Overview
### 📌 Goal of this Notebook

In this notebook, we implement **Word2Vec (Skip-Gram with Negative Sampling)** from scratch to understand how word embeddings are learned from raw text.

Rather than using prebuilt embeddings, we will:

* Prepare a real text corpus
* Build a vocabulary
* Generate (center, context) training pairs
* Implement the Skip-Gram model in PyTorch
* Train embeddings using negative sampling
* Inspect learned semantic relationships between words

This mirrors the original idea from the **Tomas Mikolov** Word2Vec paper: [*words that appear in similar contexts should have similar vector representations.*](https://arxiv.org/pdf/1301.3781)

---

### 📚 Dataset

We use **WikiText-103**, a large, clean corpus of Wikipedia articles.

Why this dataset?

* Large enough to learn meaningful word statistics
* Real natural language, not toy sentences
* Common benchmark for language modeling and embeddings

We load it through Hugging Face datasets to avoid legacy download issues from torchtext.

---

### 🧠 Intuition Behind Word2Vec (Skip-Gram)

The core idea:

> Given a **center word**, predict the **surrounding context words**.

For a sentence:

```
the cat sat on the mat
```

With window size = 2, training pairs for `"sat"` are:

```
(sat → the), (sat → cat), (sat → on), (sat → the)
```

The model learns embeddings such that words occurring in similar contexts get similar vectors.

---

### 🧱 Step 1 — Text Preprocessing

Before training, raw text must be transformed into a form the model can use:

* Lowercasing
* Tokenization into words
* Removing rare words (optional but helpful)
* Building a vocabulary: word → integer id

Why?

Neural networks operate on numbers, not text.
Vocabulary size also directly determines embedding matrix size.

---

### 🧱 Step 2 — Generating Training Pairs

We slide a window across the corpus and create:

```
(center_word_id, context_word_id)
```

This converts the entire text into millions of small supervised training examples.

This is the key trick: **we turn an unsupervised corpus into a supervised learning problem**.

---

### 🧱 Step 3 — Negative Sampling

Predicting over the full vocabulary (softmax over 50k+ words) is very expensive.

Instead, for each real pair:

* Treat the true context word as **positive**
* Sample K random words as **negative examples**

The model learns:

> Make the dot product high for real pairs, low for random pairs.

This makes training efficient and scalable.

---

### 🧱 Step 4 — Model Architecture

The Skip-Gram model is extremely simple:

* An embedding matrix for input (center words)
* An embedding matrix for output (context words)

For a pair `(center, context)`:

```
score = dot(embedding_center, embedding_context)
```

That’s it.

The power comes not from the model complexity, but from **the statistics of language**.

---

### 🧮 Step 5 — Training Objective

We use the negative sampling loss:

```
log σ(v_c · v_o) + Σ log σ(-v_c · v_n)
```

Where:

* `v_c` — center embedding
* `v_o` — true context embedding
* `v_n` — negative sample embeddings

This pushes true pairs together in vector space and random pairs apart.

---

### 🧪 Step 6 — Inspecting Learned Embeddings

After training, embeddings capture semantic structure.

We can test:

* Similar words: `similar("king") → queen, prince, throne`
* Analogies: `king - man + woman ≈ queen`

This emerges **without any labels** — purely from co-occurrence patterns.

---

### 🧭 Why This Exercise Matters

Modern NLP models (transformers, LLMs) still rely on the same fundamental idea:

> Meaning is captured by distributional context.

Word2Vec is the simplest, most elegant demonstration of this principle.

Understanding it gives intuition for:

* Embeddings
* Language modeling
* Self-supervised learning
* Why transformers work

---

### ✅ What You Should Observe

By the end:

* Frequent words have stable embeddings
* Related words cluster in vector space
* Training loss decreases smoothly
* Semantic similarity emerges naturally

This shows how structure in language can be learned without supervision.


## CBOW vs Skip-gram vs Original Word2Vec (Mikolov)

In this notebook, we implement Skip-gram model.

In [previous notebook](https://github.com/Maxstef/data-loves-ml-for-people-course/blob/main/notebooks/4_1_nlp_intro/0_3_word_embedding_basic_word2vec.ipynb) it was CBOW model.

Both are Word2Vec-style models and both introduced in the original Word2Vec framework by **Tomas Mikolov**.

### CBOW (Continuous Bag of Words)

* Predicts the **center word from surrounding context words**
* Uses a single embedding layer + linear projection
* Trained as a **full softmax classification problem**
* Computationally more expensive for large vocabularies
* Works well for frequent words and smoother representations

### Skip-gram

* Predicts **context words from a center word**
* Learns embeddings via dot products between word vectors
* Commonly trained with **negative sampling** for efficiency
* Scales better to large corpora
* Performs better on rare words

### Key Difference in Learning Objective

* **CBOW:** context → center word
* **Skip-gram:** center word → context words

### Relationship to Original Word2Vec

The original Word2Vec model includes both architectures:

* CBOW (faster, more stable)
* Skip-gram (better for rare words)

Modern implementations typically prefer Skip-gram with negative sampling due to its efficiency and strong performance on large text corpora.


## Setup & Imports

### Setup proper versions

packages used in this notebook require special package versions pairs

In [3]:
# !pip show torch torchtext

In [ ]:
# !pip uninstall -y torch torchtext torchdata

In [ ]:
# !pip install torch==2.3.1 torchtext==0.18.0 torchdata==0.7.1

In [4]:
# !pip install "portalocker>=2.0.0"

In [1]:
import os
import random
import re
from collections import Counter, OrderedDict
from dataclasses import dataclass
from time import monotonic
from typing import Dict, List, Optional, Union

import numpy as np
import torch
import torch.nn as nn
from scipy.spatial.distance import cosine
from torch.utils.data import DataLoader
# from torchtext.data import to_map_style_dataset
from torchtext.data.utils import get_tokenizer
# from torchtext.datasets import WikiText103
from tqdm import tqdm

/opt/miniconda3/envs/mlpeople5/lib/python3.11/site-packages/torchtext/data/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)


## Getting Data

The dataset loading approach shown in [the orginal article](https://towardsdatascience.com/implementing-word2vec-in-pytorch-from-the-ground-up-c7fe5bf99889/) relies on **TorchText** and a hard-coded download of **WikiText-103** from [an old S3 URL](https://s3.amazonaws.com/research.metamind.io/wikitext/wikitext-103-v1.zip) that is no longer byte-identical to the artifact expected by TorchText’s hash checker. As a result, attempts to use `torchtext.datasets.WikiText103` today often fail with MD5 mismatch or downloader errors.

Rather than fighting these legacy issues, the reliable and modern approach is to load the same corpus through **Hugging Face Datasets**, hosted on **Hugging Face**.

This gives us:

* A maintained, versioned copy of the dataset
* Robust caching and download handling
* No dependency on deprecated TorchText datapipes
* The exact same text content, delivered in a stable way

> **Note**
> This change does not alter the learning task in any way. We still use the same WikiText-103 corpus as in the article — only the data delivery mechanism has been updated to reflect the current PyTorch/NLP ecosystem.


In [2]:
# !pip install datasets

In [2]:
from datasets import load_dataset

def get_data():
    ds = load_dataset("iohadrubin/wikitext-103-raw-v1")

    train_iter = list(ds["train"]["text"])
    valid_iter = list(ds["validation"]["text"])

    return train_iter, valid_iter


train_iter, valid_iter = get_data()

for i, text in enumerate(train_iter[:6]):
    print(text)

/opt/miniconda3/envs/mlpeople5/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


= Valkyria Chronicles III =
Senjō no Valkyria 3: Unrecorded Chronicles (Japanese: 戦場のヴァルキュリア3, lit. Valkyria of the Battlefield 3), commonly referred to as Valkyria Chronicles III outside Japan, is a tactical role-playing video game developed by Sega and Media.Vision for the PlayStation Portable. Released in January 2011 in Japan, it is the third game in the Valkyria series. Employing the same fusion of tactical and real-time gameplay as its predecessors, the story runs parallel to the first game and follows the "Nameless", a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit "Calamaty Raven".
The game began development in 2010, carrying over a large portion of the work done on Valkyria Chronicles II. While it retained the standard features of the series, it also underwent multiple adjustments, such as making the game more forgiving for series newcomers. Character designer Raita Hon

## Setting Parameters and Configuration

Before building the model, we define a small set of hyperparameters that control how Word2Vec learns from the corpus.

These parameters determine:

* how much context the model sees around each word,
* how large and expressive the embedding space is,
* how many negative examples are used for efficient training,
* and how the training process is batched and optimized.

Key configuration choices:

* **Window size** — number of words to the left and right of the center word used to form training pairs.
* **Embedding dimension** — size of the vector representing each word.
* **Minimum word frequency** — filters out rare words to keep the vocabulary manageable and training stable.
* **Number of negative samples** — how many random words are contrasted against the true context word.
* **Batch size and learning rate** — control training speed and convergence behavior.
* **Number of epochs** — how many passes over the dataset we perform.

These values are not arbitrary: they directly affect the quality of learned embeddings, the training time, and the memory footprint of the model. Setting them thoughtfully ensures the model learns meaningful word relationships while remaining computationally efficient.


In [3]:
from dataclasses import dataclass, field
import torch
import torch.nn as nn

In [4]:
@dataclass(repr=True)
class Word2VecParams:

    # skipgram parameters
    MIN_FREQ = 50 
    SKIPGRAM_N_WORDS = 8
    T = 85
    NEG_SAMPLES = 50
    NS_ARRAY_LEN = 5_000_000
    SPECIALS = "<unk>"
    TOKENIZER = 'basic_english'

    # network parameters
    BATCH_SIZE = 100
    EMBED_DIM = 300
    EMBED_MAX_NORM = None
    N_EPOCHS = 5
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
    CRITERION = nn.BCEWithLogitsLoss()

In [6]:
params = Word2VecParams()
params.DEVICE

device(type='cpu')

## Building the Vocabulary

Before we can train Word2Vec, the text corpus must be converted from raw words into numerical identifiers the model can process.

In this step, we:

* tokenize the text into words,
* count how often each word appears,
* remove very rare words using a minimum frequency threshold,
* assign a unique integer index to every remaining word.

This serves two important purposes:

1. It defines the **size of the embedding matrices** — one vector per word in the vocabulary.
2. It filters noise from the dataset, since extremely rare words add parameters but contribute little useful statistical signal.

The result of this step is a mapping:

```text
word  →  index
index →  word
```

which allows us to efficiently convert the entire corpus into sequences of integers, ready for generating training pairs in the next stage.


In [7]:
class Vocab:
    def __init__(self, list, specials):
        self.stoi = {v[0]:(k, v[1]) for k, v in enumerate(list)}
        self.itos = {k:(v[0], v[1]) for k, v in enumerate(list)}
        self._specials = specials[0]
        self.total_tokens = np.nansum(
            [f for _, (_, f) in self.stoi.items()]
            , dtype=int)

    def __len__(self):
        return len(self.stoi) - 1

    def get_index(self, word: Union[str, List]):
        if isinstance(word, str):
            if word in self.stoi: 
                return self.stoi.get(word)[0]
            else:
                return self.stoi.get(self._specials)[0]
        elif isinstance(word, list):
            res = []
            for w in word:
                if w in self.stoi: 
                    res.append(self.stoi.get(w)[0])
                else:
                    res.append(self.stoi.get(self._specials)[0])
            return res
        else:
            raise ValueError(
                f"Word {word} is not a string or a list of strings."
                )


    def get_freq(self, word: Union[str, List]):
        if isinstance(word, str):
            if word in self.stoi: 
                return self.stoi.get(word)[1]
            else:
                return self.stoi.get(self._specials)[1]
        elif isinstance(word, list):
            res = []
            for w in word:
                if w in self.stoi:
                    res.append(self.stoi.get(w)[1])
                else:
                    res.append(self.stoi.get(self._specials)[1])
            return res
        else:
            raise ValueError(
                f"Word {word} is not a string or a list of strings."
                )
    

    def lookup_token(self, token: Union[int, List]):
        if isinstance(token, (int, np.int64)):
            if token in self.itos:
                return self.itos.get(token)[0]
            else:
                raise ValueError(f"Token {token} not in vocabulary")
        elif isinstance(token, list):
            res = []
            for t in token:
                if t in self.itos:
                    res.append(self.itos.get(token)[0])
                else:
                    raise ValueError(f"Token {t} is not a valid index.")
            return res

In [15]:
def yield_tokens(iterator, tokenizer):
    r = re.compile('[a-z1-9]')
    for text in iterator:
        res = tokenizer(text)
        res = list(filter(r.match, res))
        yield res
    
def build_vocab(
        iterator,
        tokenizer, 
        params: Word2VecParams,
        max_tokens: Optional[int] = None,
    ):
    counter = Counter()
    for tokens in yield_tokens(iterator, tokenizer):
        counter.update(tokens)

    # First sort by descending frequency, then lexicographically
    sorted_by_freq_tuples = sorted(
        counter.items(), key=lambda x: (-x[1], x[0])
        )

    ordered_dict = OrderedDict(sorted_by_freq_tuples)

    tokens = []
    for token, freq in ordered_dict.items():
        if freq >= params.MIN_FREQ:
            tokens.append((token, freq))

    specials = (params.SPECIALS, np.nan)
    tokens[0] = specials

    return Vocab(tokens, specials)


## Building our PyTorch Dataloaders

Once the vocabulary is defined and the corpus is converted into sequences of word indices, we prepare the data for training using **PyTorch** data loading utilities.

The goal of this step is to efficiently feed the Skip-Gram model with training examples in batches.

### What we do here:

* Generate training pairs of the form *(center word → context word)* using a sliding window over the tokenized corpus.
* Wrap these pairs in a dataset-like structure compatible with PyTorch.
* Use a DataLoader to:

  * shuffle training samples (improving generalization),
  * batch examples for efficient GPU/CPU usage,
  * streamline iteration during training.

### Why this matters:

Word2Vec training involves millions of small independent examples. Without batching and shuffling:

* training would be slow and inefficient,
* the model could overfit local patterns in the text order,
* and hardware utilization would be poor.

The DataLoader abstracts this complexity, allowing us to focus on the learning algorithm itself while ensuring scalable and efficient data pipeline execution.


In [16]:
class SkipGrams:
    def __init__(self, vocab: Vocab, params: Word2VecParams, tokenizer):
        self.vocab = vocab
        self.params = params
        self.t = self._t()
        self.tokenizer = tokenizer
        self.discard_probs = self._create_discard_dict()

    def _t(self):
        freq_list = []
        for _, (_, freq) in list(self.vocab.stoi.items())[1:]:
            freq_list.append(freq/self.vocab.total_tokens)
        return np.percentile(freq_list, self.params.T)
        

    def _create_discard_dict(self):
        discard_dict = {}
        for _, (word, freq) in self.vocab.stoi.items():
            dicard_prob = 1-np.sqrt(
                self.t / (freq/self.vocab.total_tokens + self.t))
            discard_dict[word] = dicard_prob
        return discard_dict
        

    def collate_skipgram(self, batch):
        batch_input, batch_output  = [], []
        for text in batch:
            text_tokens = self.vocab.get_index(self.tokenizer(text))

            if len(text_tokens) < self.params.SKIPGRAM_N_WORDS * 2 + 1:
                continue

            for idx in range(len(text_tokens) - self.params.SKIPGRAM_N_WORDS*2
                ):
                token_id_sequence = text_tokens[
                    idx : (idx + self.params.SKIPGRAM_N_WORDS * 2 + 1)
                    ]
                input_ = token_id_sequence.pop(self.params.SKIPGRAM_N_WORDS)
                outputs = token_id_sequence

                prb = random.random()
                del_pair = self.discard_probs.get(input_)
                if input_==0 or del_pair >= prb:
                    continue
                else:
                    for output in outputs:
                        prb = random.random()
                        del_pair = self.discard_probs.get(output)
                        if output==0 or del_pair >= prb:
                            continue
                        else:
                            batch_input.append(input_)
                            batch_output.append(output)

        batch_input = torch.tensor(batch_input, dtype=torch.long)
        batch_output = torch.tensor(batch_output, dtype=torch.long)
        
        return batch_input, batch_output


## Creating our Negative Sampling Array

To make Word2Vec training computationally efficient, we avoid computing probabilities over the entire vocabulary. Instead, we use **negative sampling**, where the model learns to distinguish true word-context pairs from randomly selected “noise” words.

As described in the Word2Vec formulation by **Tomas Mikolov**, full softmax over the vocabulary is expensive because each training step would require comparing against every word in the vocabulary. Negative sampling reduces this cost by updating only a small number of sampled “negative” words per training example.

### What we do here:

* Compute word frequencies over the corpus.
* Convert frequencies into a sampling distribution (commonly using a smoothed unigram distribution).
* Sample a small number of negative words for each training pair instead of using all non-context words.

### Why this works:

* Most words in the vocabulary are irrelevant for a given training example.
* Updating only a few negative samples significantly reduces computation.
* Frequent words are more likely to appear as negatives, but not overwhelmingly so.

### Subsampling (related but separate idea):

In addition to negative sampling, frequent words (e.g., “the”, “is”, “and”) may also be **randomly discarded during training**. This improves efficiency and reduces bias toward extremely common words, allowing the model to focus on more informative contexts.

### Result:

We obtain an efficient mechanism for selecting a small set of noise words during training. This allows the Skip-Gram model to learn meaningful vector representations without computing over the full vocabulary at each step.



In [17]:
class NegativeSampler:
    def __init__(self, vocab: Vocab, ns_exponent: float, ns_array_len: int):
        self.vocab = vocab
        self.ns_exponent = ns_exponent
        self.ns_array_len = ns_array_len
        self.ns_array = self._create_negative_sampling()

    def __len__(self):
        return len(self.ns_array)

    def _create_negative_sampling(self):

        frequency_dict = {word:freq**(self.ns_exponent) \
                          for _,(word, freq) in 
                          list(self.vocab.stoi.items())[1:]}
        frequency_dict_scaled = {
            word: 
            max(1,int((freq/self.vocab.total_tokens)*self.ns_array_len)) 
            for word, freq in frequency_dict.items()
            }
        ns_array = []
        for word, freq in tqdm(frequency_dict_scaled.items()):
            ns_array = ns_array + [word]*freq
        return ns_array

    def sample(self,n_batches: int=1, n_samples: int=1):
        samples = []
        for _ in range(n_batches):
            samples.append(random.sample(self.ns_array, n_samples))
        samples = torch.as_tensor(np.array(samples))
        return samples


## Defining the PyTorch Model

In this section, we define the Skip-gram model using **PyTorch**. The architecture is intentionally simple, reflecting the core idea behind Word2Vec: learning word representations through prediction of context relationships rather than deep network structures.

The model consists of two embedding layers:

* An **input embedding layer** that maps center words to dense vector representations.
* An **output embedding layer** that represents context words in the same vector space.

During training, a center word is passed through the input embedding layer, and its vector is compared against the embeddings of positive and negative context words using dot products. These similarity scores are then used to compute the loss for optimization.

The simplicity of this architecture is key: there are no hidden layers or nonlinear transformations. Instead, all learning happens through the adjustment of embedding vectors, allowing words that appear in similar contexts to gradually move closer together in the embedding space.

This design directly reflects the original Word2Vec idea introduced by **Tomas Mikolov**, where meaningful linguistic structure emerges purely from predicting surrounding words.


In [18]:
class Model(nn.Module):
    def __init__(self, vocab: Vocab, params: Word2VecParams):
        super().__init__()
        self.vocab = vocab
        self.t_embeddings = nn.Embedding(
            self.vocab.__len__()+1, 
            params.EMBED_DIM, 
            max_norm=params.EMBED_MAX_NORM
            )
        self.c_embeddings = nn.Embedding(
            self.vocab.__len__()+1, 
            params.EMBED_DIM, 
            max_norm=params.EMBED_MAX_NORM
            )

    def forward(self, inputs, context):
        target_embeddings = self.t_embeddings(inputs)
        n_examples = target_embeddings.shape[0]
        n_dimensions = target_embeddings.shape[1]
        target_embeddings = target_embeddings.view(n_examples, 1, n_dimensions)

        context_embeddings = self.c_embeddings(context)
        context_embeddings = context_embeddings.permute(0,2,1)

        dots = target_embeddings.bmm(context_embeddings)
        dots = dots.view(dots.shape[0], dots.shape[2])
        return dots 

    def normalize_embeddings(self):
        embeddings = list(self.t_embeddings.parameters())[0]
        embeddings = embeddings.cpu().detach().numpy() 
        norms = (embeddings ** 2).sum(axis=1) ** (1 / 2)
        norms = norms.reshape(norms.shape[0], 1)
        return embeddings / norms

    def get_similar_words(self, word, n):
        word_id = self.vocab.get_index(word)
        if word_id == 0:
            print("Out of vocabulary word")
            return

        embedding_norms = self.normalize_embeddings()
        word_vec = embedding_norms[word_id]
        word_vec = np.reshape(word_vec, (word_vec.shape[0], 1))
        dists = np.matmul(embedding_norms, word_vec).flatten()
        topN_ids = np.argsort(-dists)[1 : n + 1]

        topN_dict = {}
        for sim_word_id in topN_ids:
            sim_word = self.vocab.lookup_token(sim_word_id)
            topN_dict[sim_word] = dists[sim_word_id]
        return topN_dict

    def get_similarity(self, word1, word2):
        idx1 = self.vocab.get_index(word1)
        idx2 = self.vocab.get_index(word2)
        if idx1 == 0 or idx2 == 0:
            print("One or both words are out of vocabulary")
            return
        
        embedding_norms = self.normalize_embeddings()
        word1_vec, word2_vec = embedding_norms[idx1], embedding_norms[idx2]
 
        return cosine(word1_vec, word2_vec)

## Creating the Trainer

In this section, we define the training loop that optimizes the Skip-gram model using **PyTorch**. The trainer is responsible for orchestrating how the model learns from the generated (center, context) pairs and updates the embedding weights over time.

At a high level, the training process follows the standard Word2Vec optimization procedure:

* A batch of center words and their corresponding positive and negative context samples is passed through the model.
* The model computes similarity scores between embeddings using dot products.
* A loss function encourages high similarity for true context pairs and low similarity for negative samples.
* Gradients are computed and used to update the embedding matrices via backpropagation.

The training loop also handles:

* Iterating over multiple epochs to ensure the model sees the full corpus multiple times.
* Shuffling training data to improve generalization.
* Logging or tracking loss to monitor convergence.

This component ties together all previous steps — data preparation, vocabulary building, and model definition — and is where meaningful word representations are gradually learned through repeated exposure to contextual relationships in the corpus.


In [ ]:
class Trainer:
    def __init__(self, model: Model, params: Word2VecParams, optimizer,
                vocab: Vocab, train_iter, valid_iter, skipgrams: SkipGrams):
        self.model = model
        self.optimizer = optimizer
        self.vocab = vocab
        self.train_iter = train_iter
        self.valid_iter = valid_iter
        self.skipgrams = skipgrams
        self.params = params

        self.epoch_train_mins = {}
        self.loss = {"train": [], "valid": []}

        # sending all to device
        self.model.to(self.params.DEVICE)
        self.params.CRITERION.to(self.params.DEVICE)

        self.negative_sampler = NegativeSampler(
            vocab=self.vocab, ns_exponent=.75, 
            ns_array_len=self.params.NS_ARRAY_LEN
            )
        self.testwords = ['love', 'hurricane', 'military', 'army']
    

    def train(self):
        for epoch in range(self.params.N_EPOCHS):
            # Generate Dataloaders
            self.train_dataloader = DataLoader(
                self.train_iter,
                batch_size=self.params.BATCH_SIZE,
                shuffle=False,
                collate_fn=self.skipgrams.collate_skipgram
            )
            self.valid_dataloader = DataLoader(
                self.valid_iter,
                batch_size=self.params.BATCH_SIZE,
                shuffle=False,
                collate_fn=self.skipgrams.collate_skipgram
            )
            # training the model
            st_time = monotonic()
            self._train_epoch()
            self.epoch_train_mins[epoch] = round((monotonic()-st_time)/60, 1)

            # validating the model
            self._validate_epoch()
            print(f"""Epoch: {epoch+1}/{self.params.N_EPOCHS}\n""", 
            f"""    Train Loss: {self.loss['train'][-1]:.2}\n""",
            f"""    Valid Loss: {self.loss['valid'][-1]:.2}\n""",
            f"""    Training Time (mins): {self.epoch_train_mins.get(epoch)}"""
            """\n"""
            )
            self.test_testwords()

                
    def _train_epoch(self):
        self.model.train()
        running_loss = []

        for i, batch_data in enumerate(self.train_dataloader, 1):
            if len(batch_data[0]) == 0:
                continue
            inputs = batch_data[0].to(self.params.DEVICE)
            pos_labels = batch_data[1].to(self.params.DEVICE)
            neg_labels = self.negative_sampler.sample(
                pos_labels.shape[0], self.params.NEG_SAMPLES
                )
            neg_labels = neg_labels.to(self.params.DEVICE)
            context = torch.cat(
                [pos_labels.view(pos_labels.shape[0], 1), 
                neg_labels], dim=1
              )            

            # building the targets tensor  
            y_pos = torch.ones((pos_labels.shape[0], 1))
            y_neg = torch.zeros((neg_labels.shape[0], neg_labels.shape[1]))
            y = torch.cat([y_pos, y_neg], dim=1).to(self.params.DEVICE)

            self.optimizer.zero_grad()
            
            outputs = self.model(inputs, context)
            loss = self.params.CRITERION(outputs, y)
            loss.backward()
            self.optimizer.step()

            running_loss.append(loss.item())

        epoch_loss = np.mean(running_loss)
        
        self.loss['train'].append(epoch_loss)

    def _validate_epoch(self):
        self.model.eval()
        running_loss = []

        with torch.no_grad():
            for i, batch_data in enumerate(self.valid_dataloader, 1):
                if len(batch_data[0]) == 0:
                    continue
                inputs = batch_data[0].to(self.params.DEVICE)
                pos_labels = batch_data[1].to(self.params.DEVICE)
                neg_labels = self.negative_sampler.sample(
                    pos_labels.shape[0], self.params.NEG_SAMPLES
                    ).to(self.params.DEVICE)
                context = torch.cat(.
                    [pos_labels.view(pos_labels.shape[0], 1), 
                    neg_labels], dim=1
                  )


                # building the targets tensor  
                y_pos = torch.ones((pos_labels.shape[0], 1))
                y_neg = torch.zeros((neg_labels.shape[0], neg_labels.shape[1]))
                y = torch.cat([y_pos, y_neg], dim=1).to(self.params.DEVICE)

                preds = self.model(inputs, context).to(self.params.DEVICE)
                loss = self.params.CRITERION(preds, y)

                running_loss.append(loss.item())

            epoch_loss = np.mean(running_loss)
            self.loss['valid'].append(epoch_loss)

    def test_testwords(self, n: int = 5):
        for word in self.testwords:
            print(word)
            nn_words = self.model.get_similar_words(word, n)
            for w, sim in nn_words.items():
                print(f"{w} ({sim:.3})", end=' ')
            print('\n')

## Putting it All Together

In this final section, we combine all previously defined components into a complete Word2Vec pipeline using **PyTorch**.

At this point, we have:

* Prepared and cleaned the text corpus
* Built a vocabulary mapping words to indices
* Generated training pairs using a sliding context window
* Implemented negative sampling for efficient learning
* Defined the Skip-gram model with embedding layers
* Created a training loop to optimize the model

Now we connect everything into a single workflow:

* The dataset is passed through the DataLoader to generate batches of (center, context) examples.
* Each batch is fed into the Skip-gram model.
* The model computes loss using both positive and negative samples.
* Gradients are backpropagated to update word embeddings.
* This process is repeated over multiple epochs until convergence.

### Final Outcome

After training, each word in the vocabulary is represented as a dense vector in a continuous space. These embeddings capture semantic and syntactic relationships, meaning that words appearing in similar contexts end up close to each other in the learned vector space.

This final pipeline demonstrates how meaningful language structure can emerge purely from statistical patterns in raw text, following the original Word2Vec approach introduced by **Tomas Mikolov**.


In [21]:
params = Word2VecParams()
train_iter, valid_iter = get_data()
tokenizer = get_tokenizer(params.TOKENIZER)
vocab = build_vocab(train_iter, tokenizer, params)
skip_gram = SkipGrams(vocab=vocab, params=params, tokenizer=tokenizer)
model = Model(vocab=vocab, params=params).to(params.DEVICE)
optimizer = torch.optim.Adam(params = model.parameters())

In [25]:
vocab.get_index('python'), vocab.get_freq('python')

(14088, 391)

In [26]:
vocab.get_index('alternative'), vocab.get_freq('alternative')

(1901, 5169)

In [35]:
vocab.get_index('ukraine'), vocab.get_freq('ukraine')

(8805, 785)

In [ ]:
# trainer = Trainer(
#         model=model, 
#         params=params,
#         optimizer=optimizer, 
#         train_iter=train_iter, 
#         valid_iter=valid_iter, 
#         vocab=vocab,
#         skipgrams=skip_gram
#     )
# trainer.train()